# 07 - RecBole Hyperparameter Optimization

This notebook performs hyperparameter optimization for RecBole recommendation models.

**Models (General CF only):**
- Pop, Random, BPR, ItemKNN, EASE, LightGCN, NeuMF

**Note:** Sequential models (SASRec, GRU4Rec, BERT4Rec, SRGNN) are excluded due to a
[known RecBole bug](https://github.com/RUCAIBox/RecBole/issues/1593) with `benchmark_filename`
and `SequentialDataset`.

**Outputs:**
- `data/recbole/hpo_results/best_hyperparameters.json`

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# PyTorch 2.x compatibility fix for RecBole checkpoints
# Must be run BEFORE importing RecBole
# Uses a flag to prevent double-patching on re-runs
import torch

if not getattr(torch, "_recbole_patched", False):
    _original_load = torch.load

    def _patched_load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _original_load(*args, **kwargs)

    torch.load = _patched_load
    torch._recbole_patched = True
    print("Applied weights_only=False patch for RecBole compatibility")
else:
    print("PyTorch patch already applied (skipping)")

print(f"PyTorch version: {torch.__version__}")

In [ ]:
import json
import warnings
from itertools import product
from pathlib import Path

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_seed
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Configuration
DATA_PATH = Path("../data")
RECBOLE_DATA_PATH = DATA_PATH / "recbole"
RESULTS_PATH = DATA_PATH / "recbole" / "hpo_results"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

SEED = 42

In [ ]:
# Clear RecBole dataset cache to prevent stale state from previous runs
# RecBole caches processed datasets to disk; stale cache can cause
# checkpoint path issues and "not enough values to unpack" errors
import shutil

for cache_dir in ["dataset", "log", "saved"]:
    if Path(cache_dir).exists():
        shutil.rmtree(cache_dir)
        print(f"Cleared {cache_dir}/")
    else:
        print(f"{cache_dir}/ not found (OK)")

print("RecBole cache cleared.")

In [ ]:
def get_device() -> str:
    """Get best available device: CUDA > CPU (RecBole does not support MPS)."""
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


DEVICE = get_device()
print(f"Using device: {DEVICE}")

## Base Configuration

In [ ]:
# Base configuration for all models
BASE_CONFIG = {
    # Dataset settings
    "data_path": str(RECBOLE_DATA_PATH),
    "dataset": "redial",
    "benchmark_filename": ["train", "valid", "test"],
    # Field definitions
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "TIME_FIELD": "timestamp",
    "load_col": {
        "inter": ["user_id", "item_id", "timestamp"],
    },
    # Sequence settings (for sequential models)
    "MAX_ITEM_LIST_LENGTH": 50,
    # Training settings
    "epochs": 100,
    "train_batch_size": 256,
    "eval_batch_size": 256,
    "learning_rate": 0.001,
    "stopping_step": 10,  # Early stopping patience
    # Evaluation settings
    "eval_args": {
        "group_by": "user",
        "order": "TO",
        "split": {"LS": "valid_and_test"},
        "mode": "full",
    },
    "metrics": ["Recall", "MRR", "NDCG", "Hit", "Precision"],
    "topk": [1, 5, 10],
    "valid_metric": "NDCG@10",
    # Negative sampling (disabled for most models)
    "train_neg_sample_args": None,
    # Device and reproducibility
    "device": DEVICE,
    "seed": SEED,
    "reproducibility": True,
    "show_progress": True,
    # Logging (reduce verbosity)
    "log_wandb": False,
    "state": "INFO",
}

## Model-Specific Hyperparameter Grids

In [ ]:
# Hyperparameter search spaces from RecBole documentation
# Note: Sequential models excluded due to RecBole bug with benchmark_filename
HYPERPARAMETER_GRIDS = {
    # General CF Models
    "Pop": {},  # No hyperparameters
    "Random": {},  # No hyperparameters
    "BPR": {
        "learning_rate": [0.01, 0.005, 0.001, 0.0005, 0.0001],
    },
    "ItemKNN": {
        "k": [10, 50, 100, 200, 250, 300, 400, 500, 1000, 1500, 2000, 2500],
        "shrink": [0.0, 1.0],
    },
    "EASE": {
        "reg_weight": [1.0, 10.0, 100.0, 250.0, 500.0, 1000.0],
    },
    "LightGCN": {
        "learning_rate": [0.01, 0.005, 0.001, 0.0005, 0.0001],
        "n_layers": [1, 2, 3, 4],
        "reg_weight": [1e-05, 1e-04, 1e-03, 1e-02],
    },
    "NeuMF": {
        "learning_rate": [0.01, 0.005, 0.001, 0.0005, 0.0001],
        "dropout_prob": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
        "mlp_hidden_size": [[64, 32, 16], [32, 16, 8]],
    },
}

# Model categories (General CF only - sequential models excluded)
GENERAL_CF_MODELS = ["Pop", "Random", "BPR", "ItemKNN", "EASE", "LightGCN", "NeuMF"]

# Print grid sizes
print("Hyperparameter grid sizes:")
for model, grid in HYPERPARAMETER_GRIDS.items():
    if grid:
        n_combos = 1
        for values in grid.values():
            n_combos *= len(values)
        print(f"  {model}: {n_combos} combinations")
    else:
        print(f"  {model}: 1 (no tuning)")

print(f"\nGeneral CF models: {GENERAL_CF_MODELS}")
print(
    "\nNote: Sequential models (SASRec, GRU4Rec, BERT4Rec, SRGNN) excluded due to RecBole bug"
)

## HPO Functions

In [ ]:
def run_single_experiment(
    model_name: str,
    hyperparams: dict,
    base_config: dict,
) -> dict:
    """Run a single training experiment and return results."""
    # Merge configs
    config_dict = base_config.copy()
    config_dict.update(hyperparams)
    config_dict["model"] = model_name

    # Enable negative sampling for BPR-loss models
    if model_name in ["BPR", "LightGCN", "NeuMF"]:
        config_dict["train_neg_sample_args"] = {
            "distribution": "uniform",
            "sample_num": 1,
            "dynamic": False,
        }

    try:
        # Create config and initialize
        config = Config(model=model_name, config_dict=config_dict)
        init_seed(config["seed"], config["reproducibility"])

        # Create dataset and dataloaders
        dataset = create_dataset(config)
        train_data, valid_data, test_data = data_preparation(config, dataset)

        # Create model and trainer
        model = get_model(config["model"])(config, train_data._dataset).to(
            config["device"]
        )
        trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)

        # Train
        best_valid_score, best_valid_result = trainer.fit(
            train_data, valid_data, verbose=False, show_progress=False
        )

        # Evaluate on test (fall back to in-memory model if checkpoint missing)
        try:
            test_result = trainer.evaluate(test_data)
        except FileNotFoundError:
            test_result = trainer.evaluate(test_data, load_best_model=False)

        return {
            "status": "success",
            "best_valid_score": float(best_valid_score),
            "valid_result": {k: float(v) for k, v in best_valid_result.items()},
            "test_result": {k: float(v) for k, v in test_result.items()},
        }
    except Exception as e:
        # Print error for debugging
        print(f"    ERROR: {e}")
        import traceback

        traceback.print_exc()
        return {
            "status": "failed",
            "error": str(e),
            "best_valid_score": -float("inf"),
        }

In [ ]:
def grid_search(
    model_name: str,
    param_grid: dict,
    base_config: dict,
) -> dict:
    """Perform grid search over hyperparameter combinations."""
    if not param_grid:
        # No hyperparameters to tune
        print(f"  No hyperparameters to tune for {model_name}")
        result = run_single_experiment(model_name, {}, base_config)
        return {
            "best_params": {},
            "best_valid_score": result.get("best_valid_score", -float("inf")),
            "best_result": result,
            "all_results": [result],
        }

    # Generate all parameter combinations
    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())
    combinations = list(product(*param_values))

    print(f"  Testing {len(combinations)} parameter combinations")

    best_score = -float("inf")
    best_params = None
    best_result = None
    all_results = []

    for combo in tqdm(combinations, desc=f"  {model_name}", leave=False):
        params = dict(zip(param_names, combo))
        result = run_single_experiment(model_name, params, base_config)
        result["params"] = params
        all_results.append(result)

        if result["status"] == "success":
            score = result["best_valid_score"]
            if score > best_score:
                best_score = score
                best_params = params
                best_result = result

    return {
        "best_params": best_params,
        "best_valid_score": best_score,
        "best_result": best_result,
        "all_results": all_results,
    }

## Run HPO for All Models

In [ ]:
# Run HPO for General CF models
hpo_results = {}

print("=" * 60)
print("GENERAL CF MODELS")
print("=" * 60)

for model_name in GENERAL_CF_MODELS:
    print(f"\n{model_name}:")
    param_grid = HYPERPARAMETER_GRIDS.get(model_name, {})

    result = grid_search(model_name, param_grid, BASE_CONFIG)
    hpo_results[model_name] = result

    if result["best_params"]:
        print(f"  Best params: {result['best_params']}")
    print(f"  Best valid NDCG@10: {result['best_valid_score']:.4f}")

## Save Results

In [ ]:
# Compile best hyperparameters for each model
best_hyperparams = {}
for model_name, result in hpo_results.items():
    best_hyperparams[model_name] = {
        "params": result["best_params"] or {},
        "valid_ndcg@10": result["best_valid_score"],
        "test_result": result.get("best_result", {}).get("test_result", {}),
    }

# Save to JSON
with open(RESULTS_PATH / "best_hyperparameters.json", "w") as f:
    json.dump(best_hyperparams, f, indent=2)

print(f"Saved best hyperparameters to {RESULTS_PATH / 'best_hyperparameters.json'}")

In [ ]:
# Print summary table
print("\n" + "=" * 60)
print("HPO SUMMARY")
print("=" * 60)

print(f"\n{'Model':<15} {'Valid NDCG@10':>15} {'Best Params'}")
print("-" * 60)

# Sort by validation score
sorted_models = sorted(
    best_hyperparams.items(), key=lambda x: x[1]["valid_ndcg@10"], reverse=True
)

for model_name, hp in sorted_models:
    score = hp["valid_ndcg@10"]
    params_str = str(hp["params"]) if hp["params"] else "(default)"
    if len(params_str) > 40:
        params_str = params_str[:37] + "..."
    print(f"{model_name:<15} {score:>15.4f} {params_str}")

In [ ]:
print("\nHPO complete! Ready for final evaluation (notebook 08).")